<a href="https://colab.research.google.com/github/Savara-k/Financial-Sentiment/blob/main/Financial_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================

!pip install torch datasets scikit-learn pandas numpy
!pip install -q transformers datasets scikit-learn accelerate

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import kagglehub

# ==========================================
# 1. INSTALL GRADIO & TRANSFORMERS
# ==========================================
!pip install -q gradio transformers torch

import gradio as gr
import torch
import torch.nn.functional as F
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    pipeline, GPT2LMHeadModel, GPT2Tokenizer
)
import os


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ankurzing/sentiment-analysis-for-financial-news")

print("Path to dataset files:", path)

100%|██████████| 903k/903k [00:00<00:00, 982kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ankurzing/sentiment-analysis-for-financial-news/versions/5


In [ ]:
# ==========================================
# 2. CONFIGURATION & SEEDING
# ==========================================
MODEL_NAME = 'ProsusAI/finbert'
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 6
LEARNING_RATE = 1e-5
SEED = 42

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ==========================================
# 3. DATA LOADING & CLEANING
# ==========================================
path = kagglehub.dataset_download("ankurzing/sentiment-analysis-for-financial-news")
import glob
csv_files = glob.glob(os.path.join(path, "*.csv"))
if not csv_files: csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)

# Load
try:
    df = pd.read_csv(csv_files[0], encoding='latin-1', header=None)
    df.columns = ['label', 'text']
except Exception as e:
    print(f"Error: {e}")

# --- CRITICAL STEP: CLEANING ---
# 1. Map Labels
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['label'].astype(str).str.lower().map(label_map)
df = df.dropna()
df['label'] = df['label'].astype(int)

# 2. Remove Duplicates
initial_len = len(df)
df = df.drop_duplicates(subset=['text'])
print(f"Removed {initial_len - len(df)} duplicate rows. New size: {len(df)}")

# 3. Split Data
train_text, temp_text, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], random_state=SEED, test_size=0.2, stratify=df['label']
)
val_text, test_text, val_labels, test_labels = train_test_split(
    temp_text, temp_labels, random_state=SEED, test_size=0.5, stratify=temp_labels
)

# 4. Compute Class Weights (To handle imbalance)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights (Neg, Neu, Pos): {class_weights}")

# ==========================================
# 4. DATASET & DATALOADERS
# ==========================================
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class FinancialDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', return_token_type_ids=False,
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_data = FinancialDataset(train_text, train_labels, tokenizer, MAX_LEN)
val_data = FinancialDataset(val_text, val_labels, tokenizer, MAX_LEN)
test_data = FinancialDataset(test_text, test_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, sampler=SequentialSampler(val_data), batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, sampler=SequentialSampler(test_data), batch_size=BATCH_SIZE)

# ==========================================
# 5. MODEL & OPTIMIZER
# ==========================================
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, output_attentions=False, output_hidden_states=False
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)

# ==========================================
# 6. TRAINING WITH BEST MODEL SAVING
# ==========================================
best_val_loss = float('inf')
training_stats = []

print("\nStarting Training...")
for epoch_i in range(0, EPOCHS):
    print(f'--- Epoch {epoch_i + 1} / {EPOCHS} ---')
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        model.zero_grad()

        # Standard Forward Pass
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        logits = outputs.logits

        # Custom Weighted Loss
        loss = loss_fct(logits, b_labels)

        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION ---
    model.eval()
    total_val_loss = 0
    val_preds, val_true = [], []

    for batch in val_loader:
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        with torch.no_grad():
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
            logits = outputs.logits
            loss = loss_fct(logits, b_labels)
            total_val_loss += loss.item()

            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()
            val_preds.extend(np.argmax(logits, axis=1).flatten())
            val_true.extend(label_ids.flatten())

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = accuracy_score(val_true, val_preds)

    print(f"  Train Loss: {avg_train_loss:.3f} | Val Loss: {avg_val_loss:.3f} | Val Acc: {val_accuracy:.3f}")

    # SAVE BEST MODEL
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  -> New Best Model! Saving...")
        model.save_pretrained('./best_finbert_model')
        tokenizer.save_pretrained('./best_finbert_model')

print("\nTraining Complete!")

# ==========================================
# 7. FINAL EVALUATION (ON TEST SET)
# ==========================================
# Load the best model for evaluation
print("\nLoading Best Model for Testing...")
model = BertForSequenceClassification.from_pretrained('./best_finbert_model')
model.to(device)
model.eval()

test_preds, test_true = [], []

for batch in test_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

    logits = outputs.logits
    logits = logits.detach().cpu().numpy()
    label_ids = batch['labels'].to('cpu').numpy()

    test_preds.extend(np.argmax(logits, axis=1).flatten())
    test_true.extend(label_ids.flatten())

final_acc = accuracy_score(test_true, test_preds)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")
print(classification_report(test_true, test_preds, target_names=['Negative', 'Neutral', 'Positive']))

Device: cuda
Using Colab cache for faster access to the 'sentiment-analysis-for-financial-news' dataset.
Removed 8 duplicate rows. New size: 4838
Class Weights (Neg, Neu, Pos): [2.67080745 0.56160209 1.18348624]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]


Starting Training...
--- Epoch 1 / 6 ---


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

  Train Loss: 1.379 | Val Loss: 0.397 | Val Acc: 0.860
  -> New Best Model! Saving...
--- Epoch 2 / 6 ---
  Train Loss: 0.316 | Val Loss: 0.293 | Val Acc: 0.878
  -> New Best Model! Saving...
--- Epoch 3 / 6 ---
  Train Loss: 0.182 | Val Loss: 0.301 | Val Acc: 0.876
--- Epoch 4 / 6 ---
  Train Loss: 0.115 | Val Loss: 0.328 | Val Acc: 0.899
--- Epoch 5 / 6 ---
  Train Loss: 0.065 | Val Loss: 0.372 | Val Acc: 0.888
--- Epoch 6 / 6 ---
  Train Loss: 0.046 | Val Loss: 0.384 | Val Acc: 0.890

Training Complete!

Loading Best Model for Testing...

Final Test Accuracy: 0.8554
              precision    recall  f1-score   support

    Negative       0.78      0.93      0.85        61
     Neutral       0.94      0.83      0.88       287
    Positive       0.76      0.88      0.82       136

    accuracy                           0.86       484
   macro avg       0.83      0.88      0.85       484
weighted avg       0.87      0.86      0.86       484



In [ ]:
# ==========================================
# 2. LOAD MODELS (The "Brain")
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading models on {device}...")

# ---  Fine-Tuned Sentiment Model ---
SENTIMENT_MODEL_PATH = './best_finbert_model'
if not os.path.exists(SENTIMENT_MODEL_PATH):
    SENTIMENT_MODEL_PATH = 'ProsusAI/finbert'

print("1. Loading Sentiment Model...")
tokenizer_sent = BertTokenizer.from_pretrained(SENTIMENT_MODEL_PATH)
model_sent = BertForSequenceClassification.from_pretrained(SENTIMENT_MODEL_PATH).to(device)

# --- B. Event/Entity Extraction Model (NER) ---
print("2. Loading Entity Extraction Model...")
ner_pipeline = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    device=0 if torch.cuda.is_available() else -1,
    aggregation_strategy="simple"  # <--- THIS WAS MISSING
)

# --- C. Text Generation Model (Analyst Bot) ---
print("3. Loading Generation Model...")
gen_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gen_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# ==========================================
# 3. DEFINE PROCESSING FUNCTIONS
# ==========================================

def analyze_market_news(text):
    # --- 1. GET SENTIMENT ---
    inputs = tokenizer_sent(text, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        logits = model_sent(**inputs).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    labels = ["Negative 📉", "Neutral 😐", "Positive 📈"]
    confidences = {labels[i]: float(probs[i]) for i in range(3)}

    # --- 2. EXTRACT ENTITIES (Events) ---
    entities = ner_pipeline(text)
    extracted_orgs = list(set([e['word'] for e in entities if e['entity_group'] == 'ORG']))
    extracted_locs = list(set([e['word'] for e in entities if e['entity_group'] == 'LOC']))

    entity_text = f"**Key Players:** {', '.join(extracted_orgs) if extracted_orgs else 'None detected'}\n\n"
    entity_text += f"**Locations:** {', '.join(extracted_locs) if extracted_locs else 'None detected'}"

    # --- 3. GENERATE ANALYST COMMENTARY ---
    prompt = f"Financial news: {text}\nAnalyst outlook: This implies that the market will"
    input_ids = gen_tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Generate output
    output = gen_model.generate(
        input_ids,
        max_length=len(input_ids[0]) + 30,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        temperature=0.7,
        do_sample=True,
        pad_token_id=gen_tokenizer.eos_token_id
    )

    generated_text = gen_tokenizer.decode(output[0], skip_special_tokens=True)
    analysis = generated_text.replace(f"Financial news: {text}\n", "")

    return confidences, entity_text, analysis

# ==========================================
# 4. BUILD GRADIO UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # AI Financial Analyst & Event Extractor
    ### Multi-Model Pipeline: FinBERT (Sentiment) + BERT-NER (Entities) + GPT-2 (Generation)
    """)

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Enter Financial Headline",
                placeholder="e.g., Tesla stock falls 5% amid supply chain concerns in Shanghai.",
                lines=3
            )
            submit_btn = gr.Button("Analyze Market Signal", variant="primary")

        with gr.Column():
            # Output 1: Sentiment Gauge
            sentiment_output = gr.Label(label="Market Sentiment", num_top_classes=3)

            # Output 2: Extracted Entities
            entity_output = gr.Markdown(label="Detected Events & Entities")

            # Output 3: Generative Analysis
            gen_output = gr.Textbox(label="AI Market Outlook (Generated)", lines=3)

    submit_btn.click(
        fn=analyze_market_news,
        inputs=input_text,
        outputs=[sentiment_output, entity_output, gen_output]
    )

    gr.Examples(
        examples=[
            ["Apple reports record-breaking revenue in Q4, driving tech sector rally."],
            ["Inflation concerns rise as the Federal Reserve hints at aggressive rate hikes next month."],
            ["Google faces new antitrust lawsuit in the European Union regarding ad tech dominance."]
        ],
        inputs=input_text
    )

# Launch the App
print("Launching Gradio...")
demo.launch(share=True, debug=True)

Loading models on cuda...
1. Loading Sentiment Model...
2. Loading Entity Extraction Model...


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/tmp/ipython-input-471702722.py:75: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Launching Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c88084b895a43fe072.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
